# EXP-007 — Tối ưu siêu tham số KNN trên WESAD

**Người thực hiện:** Điền tên của bạn  
**Trạng thái:** Chưa chạy bộ dữ liệu thực tế (cần chạy R01 và R02)  
**Nhóm:** 02 — Hyperparameter Tuning

## 1. Mục tiêu và giả thuyết

Đánh giá liệu tối ưu KNN bằng **nested cross-validation theo người** có cải thiện khả năng phát hiện stress so với **EXP-003 KNN mặc định** không. Không kết luận trước khi có kết quả. Đo Macro-F1 và Balanced Accuracy trên từng subject, hai bài toán nhị phân và ba lớp.

## 2. Thiết kế thí nghiệm

- Bộ G2 được nhóm nghiệm thu: **68 đặc trưng sinh lý**, **15 người** (S2–S11, S13–S17).
- **Outer LOSO:** mỗi lần giữ 1 người test; 14 người còn lại huấn luyện.
- **Inner GroupKFold(3):** chọn siêu tham số chỉ trong 14 người train, không nhìn người test outer.
- Mỗi inner fold áp dụng SimpleImputer(median), StandardScaler và KNN **chỉ fit trên train**.
- Bốn ứng viên khóa trước: `(3, euclidean, uniform)`, `(5, manhattan, distance)`, `(9, euclidean, distance)`, `(5, euclidean, uniform)`.
- Điểm lựa chọn siêu tham số: **Macro-F1** (không dùng outer test để chọn tham số).
- Đối chứng: EXP-003 (KNN mặc định, `k=5`, Euclidean, uniform), cùng dữ liệu và outer folds.

**Lưu ý:** Các cửa sổ WESAD chồng lấn, không được tách train/test ngẫu nhiên theo cửa sổ.

## 3. Khởi tạo môi trường và đọc cấu hình

In [1]:
from pathlib import Path
import sys, json, hashlib, subprocess

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'scripts/run_experiment.py').is_file() and (p / 'src/wesad/models.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Không thấy thư mục gốc wesad-stress-detection. Mở notebook trong dự án đã giải nén.')
CONFIG_PATH = ROOT / 'experiments/02_tuning/EXP-007_KNN/config.yaml'
try:
    import yaml, pandas as pd, numpy as np, matplotlib.pyplot as plt
    import sklearn, imblearn
except ImportError as exc:
    raise ImportError(f'{exc}. Cài dependencies bằng Terminal tại ROOT: python -m pip install -r requirements.lock.txt; python -m pip install matplotlib ipykernel') from exc
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
DATA = ROOT / CONFIG['features']
SCHEMA = ROOT / CONFIG['schema']
MAIN_DIR = ROOT / CONFIG['output_dir']
QUICK_DIR = MAIN_DIR.parent / 'r03_quick_S2'
EXPORT_DIR = CONFIG_PATH.parent
print('ROOT:', ROOT)
print('Interpreter:', sys.executable)
print('Data:', DATA)
print('Output main:', MAIN_DIR)

ROOT: c:\Users\thanh\source\repos\ML\Machine-learning\wesad-stress-detection
Interpreter: c:\Users\thanh\.conda\envs\wesad31214\python.exe
Data: c:\Users\thanh\source\repos\ML\Machine-learning\wesad-stress-detection\data\features\g2_physiological68_group15_20261107\all_subjects.csv
Output main: c:\Users\thanh\source\repos\ML\Machine-learning\wesad-stress-detection\outputs\notebook_research\EXP-007\r03_main


## 4. Kiểm tra dữ liệu và checksum trước khi huấn luyện

In [2]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

if not DATA.is_file() or not SCHEMA.is_file():
    raise FileNotFoundError('Thiếu bảng G2 physiological68 hoặc feature_schema.json. Xem docs/DU_LIEU_VA_TRUY_CAP.md')
assert sha256(DATA) == CONFIG['feature_sha256'], 'Checksum dữ liệu G2 không đúng bản nhóm đã chốt'
sys.path.insert(0, str(ROOT / 'src'))
from wesad.data import load_feature_table
frame, feature_columns = load_feature_table(DATA, SCHEMA)
assert len(feature_columns) == 68
assert set(frame.subject_id) == set(CONFIG['expected_subjects'])
assert not frame.duplicated(['subject_id','window_id']).any()
assert frame.label_original.isin([1,2,3]).all()
print('Số cửa sổ:', len(frame), 'Số người:', frame.subject_id.nunique(), 'Số features:', len(feature_columns))
display(pd.DataFrame({'label': frame.label_original.value_counts().sort_index().index,
                      'count': frame.label_original.value_counts().sort_index().values}))

Số cửa sổ: 6189 Số người: 15 Số features: 68


,label,count
0,1,3370
1,2,1848
2,3,971


## 5. Kiểm tra cấu hình mô hình và grid

In [3]:
from wesad.models import build_pipeline, search_grid
from sklearn.model_selection import GroupKFold
pipe = build_pipeline('knn', CONFIG)
assert CONFIG['nested'] is True and CONFIG['balance'] is False
assert CONFIG['inner_splits'] == 3 and len(search_grid('knn', CONFIG)) == 4
print(pipe)
display(pd.DataFrame([{
    'n_neighbors': v['model__n_neighbors'][0],
    'metric': v['model__metric'][0],
    'weights': v['model__weights'][0]
} for v in CONFIG['search_spaces']['knn']]))

Pipeline(steps=[('imputer',
                 SimpleImputer(keep_empty_features=True, strategy='median')),
                ('scaler', StandardScaler()),
                ('model', KNeighborsClassifier())])


,n_neighbors,metric,weights
0,3,euclidean,uniform
1,5,manhattan,distance
2,9,euclidean,distance
3,5,euclidean,uniform


## 6. R01 — Chạy thử một fold trên S2

Bật `RUN_QUICK=True` khi đã chuẩn bị dữ liệu. Kết quả thử không được gọi là kết quả chính thức của 15 người.

In [4]:
RUN_QUICK = True

def execute(args):
    cmd = [sys.executable, 'scripts/run_experiment.py', '--config', str(CONFIG_PATH), *args]
    print('Lệnh:', ' '.join(map(str,cmd)))
    proc = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True, encoding='utf-8', errors='replace')
    print('Mã thoát:', proc.returncode)
    if proc.stdout: print('STDOUT:\n' + proc.stdout[-12000:])
    if proc.stderr: print('STDERR:\n' + proc.stderr[-18000:])
    if proc.returncode:
        raise RuntimeError('Runner thất bại. Xem STDERR phía trên để xác định nguyên nhân; không xóa output khi chưa biết lỗi.')
    return proc

if RUN_QUICK:
    execute(['--output', str(QUICK_DIR), '--outer-subjects', 'S2', '--resume'])
else:
    print('Chưa chạy R01. Đặt RUN_QUICK=True nếu muốn chạy thử S2.')

Lệnh: c:\Users\thanh\.conda\envs\wesad31214\python.exe scripts/run_experiment.py --config c:\Users\thanh\source\repos\ML\Machine-learning\wesad-stress-detection\experiments\02_tuning\EXP-007_KNN\config.yaml --output c:\Users\thanh\source\repos\ML\Machine-learning\wesad-stress-detection\outputs\notebook_research\EXP-007\r03_quick_S2 --outer-subjects S2 --resume
Mã thoát: 0
STDOUT:
multiclass/knn/S2: F1=0.4475
binary/knn/S2: F1=0.7205
Run status: partial



## 7. R02 — Chạy nested CV chính thức đủ 15 người

Cần **15 outer folds × 2 tasks = 30 fold**. Mỗi outer fold thực hiện grid search 4 ứng viên × 3 inner folds. Lệnh sẽ dùng lại các fold đã kiểm tra nếu `--resume`. Không thay đổi cấu hình sau khi đã chạy trong cùng thư mục output.

In [5]:
RUN_MAIN = True
if RUN_MAIN:
    execute(['--resume'])
else:
    print('Chưa chạy R02. Đặt RUN_MAIN=True sau khi R01 chạy ổn.')

Lệnh: c:\Users\thanh\.conda\envs\wesad31214\python.exe scripts/run_experiment.py --config c:\Users\thanh\source\repos\ML\Machine-learning\wesad-stress-detection\experiments\02_tuning\EXP-007_KNN\config.yaml --resume
Mã thoát: 0
STDOUT:
multiclass/knn/S10: F1=0.5308
multiclass/knn/S11: F1=0.5086
multiclass/knn/S13: F1=0.4804
multiclass/knn/S14: F1=0.5693
multiclass/knn/S15: F1=0.5527
multiclass/knn/S16: F1=0.6265
multiclass/knn/S17: F1=0.4498
multiclass/knn/S2: F1=0.4475
multiclass/knn/S3: F1=0.5029
multiclass/knn/S4: F1=0.4903
multiclass/knn/S5: F1=0.6927
multiclass/knn/S6: F1=0.8111
multiclass/knn/S7: F1=0.6011
multiclass/knn/S8: F1=0.6485
multiclass/knn/S9: F1=0.3391
binary/knn/S10: F1=0.8654
binary/knn/S11: F1=0.8372
binary/knn/S13: F1=0.6991
binary/knn/S14: F1=0.7716
binary/knn/S15: F1=0.7817
binary/knn/S16: F1=0.8443
binary/knn/S17: F1=0.5306
binary/knn/S2: F1=0.7205
binary/knn/S3: F1=0.9970
binary/knn/S4: F1=0.6503
binary/knn/S5: F1=0.8806
binary/knn/S6: F1=0.9971
binary/knn/S7: 

## 8. Đọc kết quả đã kiểm chứng

In [6]:
SUMMARY_PATH=MAIN_DIR/'summary.json'
READY=False
if SUMMARY_PATH.is_file():
    summary=json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
    if summary.get('status') == 'complete':
        scores=pd.read_csv(MAIN_DIR/'metrics_by_subject.csv',dtype={'subject_id':str})
        preds=pd.read_csv(MAIN_DIR/'predictions.csv',dtype={'subject_id':str,'window_id':str})
        READY=(len(scores)==30 and set(scores.subject_id)==set(CONFIG['expected_subjects'])
               and set(scores.task)=={'binary','multiclass'} and set(scores.model)=={'knn'})
        if READY:
            display(pd.DataFrame(summary['results'])[['task','model','macro_f1_mean','macro_f1_std',
                                                       'balanced_accuracy_mean','balanced_accuracy_std']])
if not READY:
    print('Chưa có kết quả R02 đầy đủ và hợp lệ. Không tự tạo các chỉ số giả.')

,task,model,macro_f1_mean,macro_f1_std,balanced_accuracy_mean,balanced_accuracy_std
0,multiclass,knn,0.550095,0.114891,0.563957,0.107271
1,binary,knn,0.786205,0.126973,0.796616,0.125982


## 9. Xuất CSV, ảnh PNG và phân tích tham số tối ưu

In [7]:
if READY:
    report_cmd=[sys.executable,'scripts/exp007_report.py','--run-dir',str(MAIN_DIR),
                '--export-dir',str(EXPORT_DIR),
                '--baseline-dir',str(ROOT/'outputs/notebook_research/EXP-003/r02_main')]
    proc=subprocess.run(report_cmd,cwd=ROOT,capture_output=True,text=True,encoding='utf-8',errors='replace')
    print(proc.stdout)
    if proc.returncode:
        print(proc.stderr)
        raise RuntimeError('Không xuất được báo cáo EXP-007. Xem thông báo lỗi trên.')
    display(pd.read_csv(EXPORT_DIR/'summary.csv'))
    display(pd.read_csv(EXPORT_DIR/'best_params_by_subject.csv').head())
else:
    print('Chỉ xuất file sau khi R02 chạy đủ 30 fold.')

Đã xuất báo cáo tại c:\Users\thanh\source\repos\ML\Machine-learning\wesad-stress-detection\experiments\02_tuning\EXP-007_KNN 
 So sánh EXP-003 và EXP-007 cùng hash dữ liệu và feature schema; xem comparison_exp003.csv.



,task,model,complete_15_subjects,n_subjects,missing_subjects,macro_f1_mean,macro_f1_std,balanced_accuracy_mean,balanced_accuracy_std,labels,confusion_matrix
0,multiclass,knn,True,15,[],0.550095,0.114891,0.563957,0.107271,"[1, 2, 3]","[[2501, 188, 681], [279, 1253, 316], [405, 302..."
1,binary,knn,True,15,[],0.786205,0.126973,0.796616,0.125982,"[0, 1]","[[3897, 444], [567, 1281]]"


,task,subject_id,n_neighbors,metric,weights
0,binary,S10,5,manhattan,distance
1,binary,S11,5,manhattan,distance
2,binary,S13,5,manhattan,distance
3,binary,S14,5,manhattan,distance
4,binary,S15,5,manhattan,distance


## 10. So sánh ghép cặp với EXP-003 (nếu có dữ liệu đối chứng tương thích)

In [8]:
comparison_file=EXPORT_DIR/'comparison_exp003.csv'
if comparison_file.exists() and READY:
    comparison=pd.read_csv(comparison_file)
    display(comparison.groupby('task')[["delta_macro_f1","delta_balanced_accuracy"]].agg(['mean','std']))
    print('Delta = EXP-007 tuned trừ EXP-003 default; số dương chỉ xu hướng cải thiện trong bộ WESAD này.')
else:
    print('Chưa có so sánh EXP-003 cùng dữ liệu. Có thể chạy EXP-003 rồi cập nhật đường dẫn baseline-dir ở cell 9.')

delta_macro_f1           delta_balanced_accuracy          
                     mean       std                    mean       std
task                                                                 
binary           0.060365  0.115413                0.042562  0.091776
multiclass      -0.005365  0.121817               -0.011302  0.120826

Delta = EXP-007 tuned trừ EXP-003 default; số dương chỉ xu hướng cải thiện trong bộ WESAD này.


## 11. Nhận xét và đánh giá kết quả thực nghiệm

### 11.1. Đánh giá phương pháp thực nghiệm

Thí nghiệm EXP-007 được xây dựng nhằm đánh giá tác động của việc tối ưu siêu tham số đối với thuật toán K-Nearest Neighbors (KNN) trong bài toán phát hiện stress trên bộ dữ liệu WESAD.

Thí nghiệm sử dụng bộ dữ liệu gồm 68 đặc trưng sinh lý của 15 người tham gia, thực hiện trên hai bài toán: phân loại nhị phân (Stress và Non-Stress) và phân loại ba lớp (Baseline, Stress và Amusement).

Phương pháp Nested Cross-Validation được áp dụng với hai cấp:

- **Outer LOSO:** Giữ lại một người để kiểm tra, sử dụng 14 người còn lại để huấn luyện.
- **Inner GroupKFold (3 folds):** Tối ưu siêu tham số trên dữ liệu huấn luyện của outer fold.

Cách tổ chức này giúp hạn chế rò rỉ dữ liệu giữa các người tham gia và bảo đảm tập kiểm tra của mỗi outer fold không tham gia vào quá trình lựa chọn siêu tham số.

### 11.2. Đánh giá quá trình tối ưu siêu tham số

Thí nghiệm xem xét bốn cấu hình KNN:

| Cấu hình | Số láng giềng (k) | Khoảng cách | Trọng số |
|---|---:|---|---|
| 1 | 3 | Euclidean | Uniform |
| 2 | 5 | Manhattan | Distance |
| 3 | 9 | Euclidean | Distance |
| 4 | 5 | Euclidean | Uniform |

Các cấu hình được đánh giá dựa trên Macro-F1 trong quá trình Inner Cross-Validation.

Việc khảo sát đồng thời số lượng láng giềng, phương pháp tính khoảng cách và cách gán trọng số cho phép nghiên cứu mức độ ảnh hưởng của từng tổ hợp tham số đối với hiệu suất phân loại.

Tuy nhiên, thí nghiệm chỉ khảo sát bốn tổ hợp được xác định trước. Vì vậy, cấu hình tốt nhất được tìm thấy chỉ là cấu hình tốt nhất trong phạm vi tìm kiếm này, không bảo đảm tối ưu trên toàn bộ không gian siêu tham số của KNN.

### 11.3. Đánh giá kết quả phân loại

Hai chỉ số được sử dụng để đánh giá là Macro-F1 và Balanced Accuracy.

Macro-F1 phản ánh khả năng phân loại tương đối đồng đều giữa các lớp, trong khi Balanced Accuracy đánh giá trung bình khả năng nhận diện đúng của các lớp, phù hợp khi số lượng mẫu giữa các trạng thái không cân bằng.

Sau khi hoàn thành 30 outer folds trên 15 người và hai bài toán, kết quả cần được tổng hợp như sau:

| Bài toán | Macro-F1 (Mean ± Std) | Balanced Accuracy (Mean ± Std) |
|---|---|---|
| Nhị phân | Chưa có kết quả | Chưa có kết quả |
| Ba lớp | Chưa có kết quả | Chưa có kết quả |

Bên cạnh giá trị trung bình, cần phân tích độ lệch chuẩn và kết quả theo từng người để đánh giá tính ổn định và khả năng tổng quát hóa của mô hình.

### 11.4. So sánh với EXP-003 — KNN Baseline

EXP-003 sử dụng cấu hình KNN mặc định với 5 láng giềng, khoảng cách Euclidean và trọng số đồng đều.

EXP-007 giữ nguyên dữ liệu, cách tiền xử lý và phương pháp đánh giá LOSO, nhưng bổ sung bước lựa chọn siêu tham số bằng Inner GroupKFold.

Mục đích của phép so sánh là xác định liệu việc tối ưu siêu tham số có cải thiện Macro-F1 và Balanced Accuracy so với KNN mặc định hay không.

Việc so sánh cần thực hiện trên cùng người kiểm tra của từng outer fold, sau đó tính chênh lệch hiệu suất giữa hai thí nghiệm.

Nếu Macro-F1 trung bình tăng, kết quả cung cấp bằng chứng về lợi ích của tối ưu tham số trong phạm vi bộ dữ liệu đã nghiên cứu. Ngược lại, nếu không có cải thiện hoặc hiệu suất giảm, điều đó cho thấy tuning không nhất thiết mang lại lợi ích trong thiết lập này.

### 11.5. Hạn chế của nghiên cứu

Thí nghiệm vẫn tồn tại một số hạn chế:

- Số lượng người tham gia WESAD tương đối nhỏ, chỉ gồm 15 người.
- Hiệu suất có thể thay đổi đáng kể do sự khác biệt sinh lý giữa các cá nhân.
- Việc tối ưu chỉ giới hạn ở bốn cấu hình KNN.
- Các cửa sổ tín hiệu có mức chồng lấn cao nên không thể xem những cửa sổ là các quan sát hoàn toàn độc lập.
- Chi phí tính toán cao hơn KNN mặc định do phải thực hiện nhiều lần huấn luyện trong Nested Cross-Validation.

### 11.6. Kết luận

EXP-007 được thiết kế nhằm nghiên cứu tác động của tối ưu siêu tham số đối với thuật toán KNN trong bài toán phát hiện stress trên WESAD.

Việc kết hợp Outer LOSO và Inner GroupKFold giúp lựa chọn siêu tham số mà không sử dụng dữ liệu của người tham gia thuộc tập kiểm tra ngoài.

Tuy nhiên, do chưa có kết quả huấn luyện chính thức được xác minh trong notebook hiện tại, chưa thể kết luận mô hình KNN sau tối ưu tốt hơn hay kém hơn EXP-003.

Sau khi có kết quả thực nghiệm, cần dựa trên Macro-F1, Balanced Accuracy, độ lệch chuẩn và chênh lệch hiệu suất theo từng người để đưa ra kết luận cuối cùng về hiệu quả của phương pháp tối ưu siêu tham số.